# Two-way interaction prediction on the role-aware line graph

In [ ]:
# ---------------- settings ----------------
import os
DATA_CSV  = "Data_msystems/Features_two_way.csv"   # adjust if the data is elsewhere
OUT_DIR   = "results"
CURV_DIR  = OUT_DIR
EPOCHS    = int(os.environ.get("EPOCHS", 3000))     # 3000 as in the paper
MODELS    = ["GAT-SAGE", "GAT", "GraphSAGE", "CGNN"]
ABLATION  = ["GAT-SAGE"]                           # models also run on the role-agnostic graph ([] to skip; not CGNN)
HIDDEN, HEADS, LR, SEED = 100, 4, 1e-3, 42
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn as dglnn
import dgl.function as fn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

device = torch.device("cpu")
if torch.cuda.is_available():
    try:
        dgl.graph(([0], [0])).to("cuda")          # does this DGL build support the GPU?
        device = torch.device("cuda")
    except Exception:
        print("WARNING: PyTorch sees a GPU, but this DGL build is CPU-only -> running on CPU (slow).")
        print("         Install the CUDA build of DGL to use the GPU (see the instructions).")
print("device:", device, "| torch", torch.__version__, "| CUDA", torch.version.cuda, "| dgl", dgl.__version__)

## 1. Data, labels, features, folds

In [ ]:
df = pd.read_csv(DATA_CSV)
df = df[df["Carbon"] != "Water"].reset_index(drop=True)

# two-way label from the signs of both directional effects (effect > 0 = positive)
pos_12 = df["1 on 2: Effect"].values > 0
pos_21 = df["2 on 1: Effect"].values > 0
y = np.where(pos_12 & pos_21, 0, np.where(~pos_12 & ~pos_21, 1, 2))   # 0 mutualism, 1 competition, 2 parasitism
CLASSES = ["Mutualism", "Competition", "Parasitism"]

# all 13 features
FEATURE_COLS = list(df.columns[6:19])
assert len(FEATURE_COLS) == 13, FEATURE_COLS
X = MinMaxScaler().fit_transform(df[FEATURE_COLS].values)

# one group per co-culture (unordered species pair + carbon): both directions stay in the same fold
groups = df.apply(lambda r: "_".join(sorted([str(r["Bug 1"]), str(r["Bug 2"])])) + "_" + str(r["Carbon"]), axis=1).values
FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups=groups))

print("nodes:", len(df), "| co-cultures:", len(set(groups)), "| features:", FEATURE_COLS)
print("class %:", {c: round(100 * np.mean(y == k), 1) for k, c in enumerate(CLASSES)})

## 2. Line graphs
**Role-aware:** two measurements are connected if, under the same carbon condition, they share the same effector (`Bug 2`) or the same affected species (`Bug 1`).

**Role-agnostic (ablation):** connected if they share any species under the same carbon condition.

In [ ]:
def pairs_within(groups_index):
    src, dst = [], []
    for idx in groups_index:
        idx = np.asarray(idx)
        if len(idx) > 1:
            u, v = np.meshgrid(idx, idx)
            keep = u != v
            src.append(u[keep]); dst.append(v[keep])
    return np.unique(np.stack([np.concatenate(src), np.concatenate(dst)], 1), axis=0)   # no duplicate edges

def make_graph(edges):
    return dgl.graph((torch.tensor(edges[:, 0]), torch.tensor(edges[:, 1])), num_nodes=len(df))

# role-aware: same effector or same affected, same carbon
EDGES = pairs_within([idx for role in ["Bug 2", "Bug 1"] for idx in df.groupby([role, "Carbon"]).indices.values()])

# role-agnostic: share any species, same carbon
long = pd.concat([pd.DataFrame({"node": df.index, "species": df["Bug 1"], "Carbon": df["Carbon"]}),
                  pd.DataFrame({"node": df.index, "species": df["Bug 2"], "Carbon": df["Carbon"]})]).drop_duplicates()
EDGES_AG = pairs_within(long.groupby(["species", "Carbon"])["node"].apply(np.array).values)

g = make_graph(EDGES)
GRAPHS = {"role-aware": g, "role-agnostic": make_graph(EDGES_AG)}
for k, gk in GRAPHS.items():
    print(f"{k:14s}: nodes {gk.num_nodes()} | directed edges {gk.num_edges()} | undirected {gk.num_edges() // 2}")

## 3. Ollivier–Ricci curvature (for CGNN only)
Computed once and saved to `results/curvature_two-way.csv`; later runs reuse the file.

In [ ]:
CURV_FILE = f"{CURV_DIR}/curvature_two-way.csv"
if "CGNN" in MODELS:
    if os.path.exists(CURV_FILE):
        curv = pd.read_csv(CURV_FILE)
    else:
        try:
            from GraphRicciCurvature.OllivierRicci import OllivierRicci
        except ImportError:
            raise ImportError("Install it first:  pip install --user GraphRicciCurvature  (then restart the kernel)")
        import networkx as nx
        t0 = time.time()
        G = nx.Graph()
        G.add_nodes_from(range(len(df)))
        G.add_edges_from(map(tuple, EDGES[EDGES[:, 0] < EDGES[:, 1]]))
        orc = OllivierRicci(G, alpha=0.5, verbose="ERROR")
        orc.compute_ricci_curvature()
        curv = pd.DataFrame([(u, v, d["ricciCurvature"]) for u, v, d in orc.G.edges(data=True)],
                            columns=["u", "v", "curvature"])
        curv.to_csv(CURV_FILE, index=False)
        print(f"curvature computed in {time.time() - t0:.0f}s")
    lookup = {}
    for u, v, c in curv[["u", "v", "curvature"]].itertuples(index=False):
        lookup[(int(u), int(v))] = c; lookup[(int(v), int(u))] = c
    r = np.array([lookup.get((int(u), int(v)), 0.0) for u, v in EDGES])
    deg = np.zeros(len(df)); np.add.at(deg, EDGES[:, 1], r)
    deg = np.where(deg == 0, 1.0, deg)
    g.edata["w"] = torch.tensor(r / np.sqrt(np.abs(deg[EDGES[:, 0]] * deg[EDGES[:, 1]])), dtype=torch.float32)
    print("curvature: mean", round(r.mean(), 4), "| min", round(r.min(), 4), "| max", round(r.max(), 4))

## 4. Models (two layers, hidden size 100, as in the paper)

In [ ]:
class GAT_SAGE(nn.Module):                # GAT layer (4 heads) -> GraphSAGE layer
    def __init__(s, i, h, o, heads=HEADS):
        super().__init__()
        s.gat1 = dglnn.GATConv(i, h, num_heads=heads, activation=F.relu)
        s.conv2 = dglnn.SAGEConv(h * heads, o, aggregator_type="mean")
    def forward(s, g, x, return_hidden=False):
        h = s.gat1(g, x).flatten(1)
        out = s.conv2(g, h)
        return (out, h) if return_hidden else out

class GAT(nn.Module):                     # two GAT layers
    def __init__(s, i, h, o, heads=HEADS):
        super().__init__()
        s.gat1 = dglnn.GATConv(i, h, num_heads=heads, activation=F.elu)
        s.gat2 = dglnn.GATConv(h * heads, o, num_heads=1)
    def forward(s, g, x, return_hidden=False):
        h = s.gat1(g, x).flatten(1)
        out = s.gat2(g, h).mean(1)
        return (out, h) if return_hidden else out

class SAGE(nn.Module):                    # two GraphSAGE layers (mean aggregation)
    def __init__(s, i, h, o):
        super().__init__()
        s.conv1 = dglnn.SAGEConv(i, h, aggregator_type="mean")
        s.conv2 = dglnn.SAGEConv(h, o, aggregator_type="mean")
    def forward(s, g, x, return_hidden=False):
        h = F.relu(s.conv1(g, x))
        out = s.conv2(g, h)
        return (out, h) if return_hidden else out

class CGNNLayer(nn.Module):               # curvature-weighted message passing
    def __init__(s, i, o):
        super().__init__(); s.linear = nn.Linear(i, o)
    def forward(s, g, x):
        with g.local_scope():
            g.ndata["h"] = x
            g.update_all(lambda e: {"m": e.data["w"].unsqueeze(1) * e.src["h"]}, fn.sum("m", "h_neigh"))
            return s.linear(g.ndata["h_neigh"])

class CGNN(nn.Module):
    def __init__(s, i, h, o):
        super().__init__(); s.l1 = CGNNLayer(i, h); s.l2 = CGNNLayer(h, o)
    def forward(s, g, x, return_hidden=False):
        h = F.relu(s.l1(g, x))
        out = s.l2(g, h)
        return (out, h) if return_hidden else out

BUILD = {"GAT-SAGE": lambda: GAT_SAGE(13, HIDDEN, 3), "GAT": lambda: GAT(13, HIDDEN, 3),
         "GraphSAGE": lambda: SAGE(13, HIDDEN, 3), "CGNN": lambda: CGNN(13, HIDDEN, 3)}

## 5. Grouped 5-fold cross-validation (metrics on validation nodes only)

In [ ]:
def per_fold_metrics(name, k, true, pred):
    r = {"model": name, "fold": k, "accuracy": accuracy_score(true, pred),
         "f1_macro": f1_score(true, pred, average="macro", zero_division=0)}
    for fn_, nm in [(precision_score, "precision"), (recall_score, "recall"), (f1_score, "f1")]:
        for c, v in zip(CLASSES, fn_(true, pred, average=None, labels=[0, 1, 2], zero_division=0)):
            r[f"{nm}_{c}"] = v
    return r

def run_gnn(name, gname="role-aware"):
    gd, xd, yd = GRAPHS[gname].to(device), torch.tensor(X, dtype=torch.float32, device=device), torch.tensor(y, device=device)
    rows, cm, emb = [], np.zeros((3, 3), dtype=int), []
    for k, (tr, va) in enumerate(FOLDS, start=1):
        t0 = time.time()
        torch.manual_seed(SEED + k)
        tm = torch.zeros(len(y), dtype=torch.bool, device=device); tm[tr] = True
        vm = torch.zeros(len(y), dtype=torch.bool, device=device); vm[va] = True
        model = BUILD[name]().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        for ep in range(EPOCHS):
            model.train()
            loss = F.cross_entropy(model(gd, xd)[tm], yd[tm])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            out, hid = model(gd, xd, return_hidden=True)
            pred = out[vm].argmax(1).cpu().numpy()
        true = y[va]                                          # VALIDATION nodes only
        rows.append(per_fold_metrics(name, k, true, pred))
        cm += confusion_matrix(true, pred, labels=[0, 1, 2])
        emb.append((hid[vm].cpu().numpy().astype(np.float16), true, np.full(len(true), k)))
        print(f"  {name} [{gname}] fold {k}: macro-F1 {rows[-1]['f1_macro']:.4f} | mutualism F1 {rows[-1]['f1_Mutualism']:.4f} | {time.time()-t0:.0f}s")
    np.savez_compressed(f"{OUT_DIR}/embeddings_two-way_{name}_{gname}.npz", h=np.concatenate([e[0] for e in emb]),
                        label=np.concatenate([e[1] for e in emb]), fold=np.concatenate([e[2] for e in emb]))
    return pd.DataFrame(rows), cm

RESULTS, CMS, ABL = {}, {}, {}
for name in MODELS:
    print(f"\n=== {name} ===")
    RESULTS[name], CMS[name] = run_gnn(name)
    RESULTS[name].to_csv(f"{OUT_DIR}/folds_two-way_{name}_role-aware_grouped.csv", index=False)
for name in ABLATION:
    print(f"\n=== {name} (role-agnostic) ===")
    ABL[name], CMS[f"{name} (role-agnostic)"] = run_gnn(name, "role-agnostic")
    ABL[name].to_csv(f"{OUT_DIR}/folds_two-way_{name}_role-agnostic_grouped.csv", index=False)

## 7. Summary (mean ± std over the 5 folds)

In [ ]:
order = [m for m in ["GAT-SAGE", "GAT", "GraphSAGE", "CGNN"] if m in RESULTS]
metrics = [f"{m}_{c}" for c in CLASSES for m in ["precision", "recall", "f1"]] + ["f1_macro", "accuracy"]
summary = pd.DataFrame({m: [f"{RESULTS[m][c].mean():.4f} ± {RESULTS[m][c].std(ddof=0):.4f}" for c in metrics] for m in order},
                       index=metrics)
for name in ABL:
    summary[f"{name} (role-agnostic)"] = [f"{ABL[name][c].mean():.4f} ± {ABL[name][c].std(ddof=0):.4f}" for c in metrics]
summary.to_csv(f"{OUT_DIR}/table2_two-way_summary.csv")
summary

## 8. Table 2 in LaTeX (best value per row in bold)

In [ ]:
def cell(m, c, best):
    s = f"{RESULTS[m][c].mean():.4f} $\\pm$ {RESULTS[m][c].std(ddof=0):.4f}"
    return f"\\textbf{{{s}}}" if m == best else s

lines = [r"\begin{table*}[t]", r"\centering", r"\small",
         r"\caption{Classification performance for two-way microbial interactions of role-aware GNN architectures "
         r"(mean $\pm$ standard deviation, $n = 5$ folds). The best value in each row is shown in bold.}",
         r"\label{tab:metrics_vs_methods}",
         r"\begin{tabular}{ll" + "c" * len(order) + "}", r"\toprule",
         "Class & Metric & " + " & ".join(order) + r" \\", r"\midrule"]
signs = {"Mutualism": "$(+,+)$", "Competition": "$(-,-)$", "Parasitism": "$(+,-)$"}
for c in CLASSES:
    for i, (m, label) in enumerate([("precision", "Precision"), ("recall", "Recall"), ("f1", "F1-score")]):
        col = f"{m}_{c}"
        best = max(order, key=lambda mm: RESULTS[mm][col].mean())
        first = rf"\multirow{{3}}{{*}}{{{c} {signs[c]}}}" if i == 0 else ""
        lines.append(f"{first} & {label} & " + " & ".join(cell(mm, col, best) for mm in order) + r" \\")
    lines.append(r"\midrule")
for col, label in [("f1_macro", "Macro F1-score"), ("accuracy", "Accuracy")]:
    best = max(order, key=lambda mm: RESULTS[mm][col].mean())
    lines.append(rf"\multicolumn{{2}}{{l}}{{{label}}} & " + " & ".join(cell(mm, col, best) for mm in order) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
latex = "\n".join(lines)
open(f"{OUT_DIR}/table2_two-way.tex", "w").write(latex)
print(latex)

## 9. Paired t-tests (same folds): GAT-SAGE vs the other GNNs, role-aware vs role-agnostic

In [ ]:
from scipy.stats import ttest_rel
def compare(a, b, la, lb):
    for col in ["f1_macro", "f1_Mutualism"]:
        t, p = ttest_rel(a[col], b[col])
        print(f"  {la:24s} vs {lb:14s} {col:13s}: diff {a[col].mean() - b[col].mean():+.4f}  p = {p:.4f}")
def gnn_tests(R, A):
    if "GAT-SAGE" in R:
        for m in [x for x in R if x != "GAT-SAGE"]:
            compare(R["GAT-SAGE"], R[m], "GAT-SAGE", m)
    for name in A:
        compare(R[name], A[name], f"{name} role-aware", "role-agnostic")
gnn_tests(RESULTS, ABL)

## 10. Confusion matrices of all models, normalised by the total
All nine cells together sum to 100%; the number in brackets is the count summed over the 5 validation folds. Two rows; saved as `confusion_two-way.pdf`.

In [ ]:
import matplotlib.pyplot as plt
panels = [p for m in order for p in ([m] + [f"{m} (role-agnostic)" for n in ABL if n == m])]   # all models + ablation
ncols = (len(panels) + 1) // 2                                                     # 2 rows
fig, axes = plt.subplots(2, ncols, figsize=(4.6 * ncols, 9.4), squeeze=False)
for ax, m in zip(axes.flat, panels):
    cm = CMS[m] / CMS[m].sum() * 100              # normalised by the total: all 9 cells sum to 100%
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:.1f}%\n({CMS[m][i, j]})", ha="center", va="center", fontsize=11,
                    color="white" if cm[i, j] > 0.6 * cm.max() else "black")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(["Mut", "Comp", "Para"], fontsize=11); ax.set_yticklabels(["Mut", "Comp", "Para"], fontsize=11)
    ax.set_title(m, fontsize=14); ax.set_xlabel("Predicted", fontsize=12)
for ax in axes.flat[len(panels):]:                                                 # hide the empty panel if any
    ax.axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel("True", fontsize=12)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_two-way.pdf", bbox_inches="tight"); plt.show()

## 11. t-SNE of all models (validation nodes only)
Hidden-layer embeddings of all GNNs. Two rows; saved as `tsne_two-way.pdf`.

In [ ]:
from sklearn.manifold import TSNE
ncols = (len(panels) + 1) // 2                                                     # 2 rows
fig, axes = plt.subplots(2, ncols, figsize=(4.6 * ncols, 9.4), squeeze=False)
for ax, m in zip(axes.flat, panels):
    name, gname = (m.replace(" (role-agnostic)", ""), "role-agnostic") if m.endswith("(role-agnostic)") else (m, "role-aware")
    d = np.load(f"{OUT_DIR}/embeddings_two-way_{name}_{gname}.npz")
    h_all, y_all, title = d["h"], d["label"], m
    idx = np.random.default_rng(SEED).choice(len(y_all), size=min(4000, len(y_all)), replace=False)
    z = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(h_all[idx].astype(np.float32))
    lbl = y_all[idx]
    for l_, col, txt in [(1, "#9aa7b5", "competition"), (2, "#e8a33d", "parasitism"), (0, "#d0312d", "mutualism")]:
        ax.scatter(z[lbl == l_, 0], z[lbl == l_, 1], s=5, c=col, label=txt, alpha=0.6)
    ax.set_title(title, fontsize=14); ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[len(panels):]:                                                 # hide the empty panel if any
    ax.axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, markerscale=5, frameon=False, fontsize=14, bbox_to_anchor=(0.5, -0.03))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/tsne_two-way.pdf", bbox_inches="tight"); plt.show()